In [2]:
import pandas as pd
import numpy as np
import re
from datetime import time
import pytz
import random
import ast
import os
from html import unescape
import json
import math
import matplotlib.pyplot as plt


## 1-Remove retweets

In [3]:
df = pd.read_csv("tweets.csv")
df = df[df["isRetweet"] == "f"]


## 2-Construct event_date

In [4]:
# 1. Filter tweets to 2016-2019 
df['datetime_raw'] = pd.to_datetime(df['date'], utc=True, errors='coerce')


start = pd.Timestamp('2016-01-01T00:00:00', tz='UTC')
end   = pd.Timestamp('2019-12-31T23:59:59', tz='UTC')

before_count = len(df)
df = df[(df['datetime_raw'] >= start) & (df['datetime_raw'] <= end)].copy()
after_count = len(df)
print(f"Filtered rows: before={before_count}, after={after_count}")

df['event_minute'] = df['datetime_raw'].dt.floor('min')
df['event_minute'] = pd.to_datetime(df['event_minute'], utc=True)
print(df['event_minute'].dtype)

Filtered rows: before=46694, after=14369
datetime64[ns, UTC]


## 3- company dictionary

In [5]:
data = [
# Technology & Internet
("AAPL","Apple Inc.","Technology & Internet",
 "Apple Inc|Apple Incorporated|Apple"),

("MSFT","Microsoft Corporation","Technology & Internet",
 "Microsoft Corporation|Microsoft"),

("GOOGL","Alphabet Inc.","Technology & Internet",
 "Alphabet Inc|Alphabet|Google|Google LLC"),

("META","Meta Platforms, Inc.","Technology & Internet",
 "Meta Platforms|Meta|Facebook"),

("AMZN","Amazon.com, Inc.","Technology & Internet",
 "Amazon.com|Amazon|Amazon Web Services|AWS"),


#  Oil & Natural Gas
("XOM","Exxon Mobil Corporation","Oil & Natural Gas",
 "Exxon Mobil|Exxon Mobil Corporation|Exxon"),

("CVX","Chevron Corporation","Oil & Natural Gas",
 "Chevron Corporation|Chevron"),

("COP","ConocoPhillips","Oil & Natural Gas",
 "ConocoPhillips|Conoco Phillips"),

("OXY","Occidental Petroleum","Oil & Natural Gas",
 "Occidental Petroleum|Occidental"),

("BP","BP plc","Oil & Natural Gas",
 "BP plc|BP"),


#  Semiconductors
("NVDA","NVIDIA Corporation","Semiconductors",
 "NVIDIA Corporation|NVIDIA|Nvidia"),

("AMD","Advanced Micro Devices","Semiconductors",
 "Advanced Micro Devices|AMD"),

("INTC","Intel Corporation","Semiconductors",
 "Intel Corporation|Intel"),

("AVGO","Broadcom Inc.","Semiconductors",
 "Broadcom|Broadcom Inc"),

("QCOM","Qualcomm Incorporated","Semiconductors",
 "Qualcomm|Qualcomm Inc"),


#  Chemicals
("DOW","Dow Inc.","Chemicals",
 "Dow Inc|Dow Incorporated"),

("DD","DuPont de Nemours","Chemicals",
 "DuPont|DuPont de Nemours"),

("LIN","Linde plc","Chemicals",
 "Linde plc|Linde"),

("APD","Air Products and Chemicals","Chemicals",
 "Air Products and Chemicals|Air Products"),

("ECL","Ecolab Inc.","Chemicals",
 "Ecolab|Ecolab Inc"),


#  Financials (Banks)
("JPM","JPMorgan Chase","Financials (Banks)",
 "JPMorgan Chase|JPMorgan"),

("BAC","Bank of America","Financials (Banks)",
 "Bank of America|BofA"),

("C","Citigroup Inc.","Financials (Banks)",
 "Citigroup|Citigroup Inc"),

("MS","Morgan Stanley","Financials (Banks)",
 "Morgan Stanley"),

("GS","Goldman Sachs","Financials (Banks)",
 "Goldman Sachs"),


#  Construction & Infrastructure
("CAT","Caterpillar Inc.","Construction & Infrastructure",
 "Caterpillar Inc|Caterpillar"),

("VMC","Vulcan Materials","Construction & Infrastructure",
 "Vulcan Materials"),

("MLM","Martin Marietta","Construction & Infrastructure",
 "Martin Marietta"),

("J","Jacobs Solutions","Construction & Infrastructure",
 "Jacobs Solutions|Jacobs Engineering"),

("PWR","Quanta Services","Construction & Infrastructure",
 "Quanta Services"),


# Defense & Aerospace
("LMT","Lockheed Martin","Defense & Aerospace",
 "Lockheed Martin|Lockheed"),

("NOC","Northrop Grumman","Defense & Aerospace",
 "Northrop Grumman|Northrop"),

("RTX","Raytheon Technologies","Defense & Aerospace",
 "Raytheon Technologies|Raytheon"),

("GD","General Dynamics","Defense & Aerospace",
 "General Dynamics"),

("BA","Boeing Company","Defense & Aerospace",
 "Boeing|Boeing Company"),


# Automobiles & Auto Components
("GM","General Motors","Automobiles & Auto Components",
 "General Motors|General Motors Company"),

("TSLA","Tesla Inc.","Automobiles & Auto Components",
 "Tesla Inc|Tesla|Tesla Motors"),

("CMI","Cummins Inc.","Automobiles & Auto Components",
 "Cummins Inc|Cummins"),

("BWA","BorgWarner","Automobiles & Auto Components",
 "BorgWarner|BorgWarner Inc"),

("APTV","Aptiv PLC","Automobiles & Auto Components",
 "Aptiv|Aptiv PLC"),


#  News & Publishing
("FOXA","Fox Corporation","News & Publishing",
 "Fox Corporation|Fox News"),

("NYT","The New York Times Company","News & Publishing",
 "New York Times|The New York Times|NYT"),

("GCI","Gannett Co., Inc.","News & Publishing",
 "Gannett"),

("SBGI","Sinclair Broadcast Group","News & Publishing",
 "Sinclair Broadcast Group|Sinclair"),

("IHRT","iHeartMedia, Inc.","News & Publishing",
 "iHeartMedia|iHeartRadio"),


#  Airlines & Travel
("DAL","Delta Air Lines","Airlines & Travel",
 "Delta Air Lines"),

("AAL","American Airlines Group","Airlines & Travel",
 "American Airlines"),

("UAL","United Airlines Holdings","Airlines & Travel",
 "United Airlines"),

("LUV","Southwest Airlines","Airlines & Travel",
 "Southwest Airlines"),

("MAR","Marriott International","Airlines & Travel",
 "Marriott International|Marriott"),
]

df_dict = pd.DataFrame(
    data,
    columns=["ticker","company_name","industry","aliases"]
)

df_dict.to_csv("company_dict_10_industries.csv", index=False)

print("10-industry dictionary saved as company_dict_10_industries.csv")
print("Total firms:", len(df_dict))
print("Industries:", df_dict['industry'].unique())


10-industry dictionary saved as company_dict_10_industries.csv
Total firms: 50
Industries: ['Technology & Internet' 'Oil & Natural Gas' 'Semiconductors' 'Chemicals'
 'Financials (Banks)' 'Construction & Infrastructure'
 'Defense & Aerospace' 'Automobiles & Auto Components' 'News & Publishing'
 'Airlines & Travel']


In [6]:
# -----------------------------
# CONFIG
# -----------------------------
DICT_PATH = "company_dict_10_industries.csv"   # company dictionary (csv)
DF_OUT_ANNOTATED = "tweets_matched.csv"

CASHTAG_ONLY_TICKERS = set(["C", "BP"])  
AUDIT_SAMPLE_FRAC = 0.02                 # sample 2% for audit
LOW_CONF_SAMPLE_MAX = 500                # low-conf

# ---- add case-sensitive alias list here ----
# Put aliases that are high-risk common nouns and should only match with exact case.
# Example: "Apple" to avoid matching "apple pie" / "bite the apple".
CASE_SENSITIVE_ALIASES = {"Apple", "Dow", "Meta"}  

# -----------------------------
# helper utilities
# -----------------------------
def parse_list_field(x):
    if pd.isna(x) or x=="":
        return []
    s = str(x).strip()
    if s.startswith("[") and s.endswith("]"):
        try:
            import ast
            v = ast.literal_eval(s)
            if isinstance(v, (list,tuple)):
                return [str(e) for e in v]
        except:
            pass
    if "|" in s:
        return [p.strip() for p in s.split("|") if p.strip()]
    if "," in s:
        return [p.strip().strip("'\"") for p in s.split(",") if p.strip()]
    return [s]

def is_quoted_tweet(text):
    if not isinstance(text, str):
        return False
    t = text.strip()
    if t.startswith("RT @") or re.match(r'^@[\w_]+:', t) or re.match(r'^"@[\w_]+:', t):
        return True
    return False

def token_list(s):
    return re.findall(r"\w+", str(s).lower())

def ceo_cooccurs_with_company(raw_text, ceo_name, company_name, max_word_distance=6):
    if not ceo_name or not company_name:
        return False
    toks = token_list(raw_text)
    ceo_toks = token_list(ceo_name)
    comp_toks = token_list(company_name)
    if not ceo_toks or not comp_toks:
        return False
    pos_ceo = [i for i in range(len(toks)) if toks[i:i+len(ceo_toks)] == ceo_toks]
    pos_comp = [i for i in range(len(toks)) if toks[i:i+len(comp_toks)] == comp_toks]
    for p in pos_ceo:
        for q in pos_comp:
            if abs(p-q) <= max_word_distance:
                return True
    return False

# -----------------------------
# load company dictionary
# -----------------------------
if not os.path.exists(DICT_PATH):
    raise FileNotFoundError(f"Dictionary not found at {DICT_PATH}. Please place your CSV there.")

df_dict = pd.read_csv(DICT_PATH, dtype=str).fillna("")
df_dict['ticker'] = df_dict['ticker'].str.strip().str.upper()
df_dict['company_name'] = df_dict['company_name'].str.strip()
if 'aliases' not in df_dict.columns:
    df_dict['aliases'] = ""

# -----------------------------
# build alias rows
# -----------------------------
alias_rows = []
for _, r in df_dict.iterrows():
    t = r['ticker']
    cname = r['company_name']
    ind = r.get('industry','') or ''
    if cname:
        alias_rows.append((t, cname, ind, cname, cname.strip().lower()))
    aliases_list = parse_list_field(r.get('aliases', ''))
    for a in aliases_list:
        a = a.strip()
        if a:
            alias_rows.append((t, cname, ind, a, a.lower()))

# deduplicate (ticker + normalized alias) and sort by alias length desc
seen = set()
alias_rows_unique = []
for t,cname,ind,a_orig,a_norm in alias_rows:
    key = (t, a_norm)
    if key not in seen:
        alias_rows_unique.append((t,cname,ind,a_orig,a_norm))
        seen.add(key)
alias_rows_unique.sort(key=lambda x: len(x[4]), reverse=True)

# -----------------------------
# compile alias patterns (robust boundaries)
# -----------------------------
alias_patterns = []
for t,cname,ind,a_orig,a_norm in alias_rows_unique:
    a_norm_clean = re.sub(r"\s+", " ", a_norm.strip())
    a_norm_clean = a_norm_clean.strip(" '\".,:;!-()[]{}")
    alias_original = a_orig.strip()
    try:
        # If alias_original is in CASE_SENSITIVE_ALIASES, compile a case-sensitive pattern using the original alias text.
        if alias_original in CASE_SENSITIVE_ALIASES:
            # case-sensitive: do NOT use IGNORECASE flag
            pat = re.compile(r'(?<!\w)' + re.escape(alias_original) + r'(?!\w)')
        else:
            # default: case-insensitive matching
            pat = re.compile(r'(?<!\w)' + re.escape(a_norm_clean) + r'(?!\w)', flags=re.IGNORECASE)
        alias_patterns.append((pat, t, cname, ind, a_orig, a_norm_clean))
    except re.error:
        alias_patterns.append((None, t, cname, ind, a_orig, a_norm_clean))
# ensure None patterns go last
alias_patterns.sort(key=lambda x: (0 if x[0] is None else len(x[5])), reverse=True)

# cashtag regex
CASHTAG_RE = re.compile(r'\$([A-Za-z]{1,5})(?=\W|$)')

# -----------------------------
# matching function
# -----------------------------
def match_companies_in_text(text, alias_patterns, df_dict_local=None, ticker_set=None, cashtag_only_set=None):
    """
    返回： (matched_tickers, match_method, matched_aliases, matched_industries, is_quoted)
    match_method in {'cashtag','name','alias','ceo_cooccur','none'}
    """
    if not isinstance(text, str):
        return [], 'none', [], [], False

    raw_text = unescape(text)
    text_low = raw_text.lower()
    is_quoted = is_quoted_tweet(raw_text)

    matched = []
    matched_aliases = []
    matched_inds = []

    # 1) cashtag priority
    cashtags = CASHTAG_RE.findall(raw_text)
    for tag in cashtags:
        tag_up = tag.strip().upper()
        if (ticker_set is None or tag_up in ticker_set):
            if tag_up not in matched:
                matched.append(tag_up)
                matched_aliases.append(f"${tag_up}")
                if df_dict_local is not None:
                    matched_inds.append(df_dict_local.loc[df_dict_local['ticker']==tag_up,'industry'].iloc[0] if tag_up in df_dict_local['ticker'].values else "")
                else:
                    matched_inds.append("")
    if matched:
        return matched, 'cashtag', matched_aliases, matched_inds, is_quoted

    # 2) alias/name matching
    for pat, t, cname, ind, a_orig, a_norm in alias_patterns:
        if cashtag_only_set and t in cashtag_only_set:
            continue
        found = False
        if pat is None:
            # substring fallback uses lowercased normalized alias
            idx = text_low.find(a_norm)
            if idx >= 0:
                found = True
        else:
            m = pat.search(raw_text)
            if m:
                found = True
        if found and t not in matched:
            matched.append(t)
            matched_aliases.append(a_orig)
            matched_inds.append(ind)
    if matched:
        match_method = 'alias'
        if df_dict_local is not None:
            for mt, ma in zip(matched, matched_aliases):
                cname_row = df_dict_local.loc[df_dict_local['ticker']==mt,'company_name']
                if not cname_row.empty and cname_row.iloc[0].strip().lower() == ma.strip().lower():
                    match_method = 'name'
                    break
        return matched, match_method, matched_aliases, matched_inds, is_quoted

    # 3) CEO co-occurrence 
    if df_dict_local is not None and 'ceo_2016_2019' in df_dict_local.columns:
        for _, row in df_dict_local.iterrows():
            ticker = row['ticker']
            ceo_name = row.get('ceo_2016_2019', "")
            if not ceo_name:
                continue
            if ceo_name.lower() in text_low:
                if ceo_cooccurs_with_company(raw_text, ceo_name, row['company_name'], max_word_distance=6):
                    if ticker not in matched:
                        matched.append(ticker)
                        matched_aliases.append(f"CEO:{ceo_name}")
                        matched_inds.append(row.get('industry',''))
        if matched:
            return matched, 'ceo_cooccur', matched_aliases, matched_inds, is_quoted

    # none
    return [], 'none', [], [], is_quoted

# -----------------------------
# extras: find match positions & contexts for audit
# -----------------------------
def extract_match_extras(text, matched_tickers, alias_rows_unique_local):
    extras = {"positions": [], "contexts": []}
    if not matched_tickers:
        return extras
    rt = text
    tlow = rt.lower()
    for tkr in matched_tickers:
        for (_t, cname, ind, a_orig, a_norm) in alias_rows_unique_local:
            if _t != tkr:
                continue
            alias_original = a_orig.strip()
            try:
                # Use case-sensitive compilation if alias_original is in CASE_SENSITIVE_ALIASES
                if alias_original in CASE_SENSITIVE_ALIASES:
                    pat = re.compile(r'(?<!\w)' + re.escape(alias_original) + r'(?!\w)')
                else:
                    pat = re.compile(r'(?<!\w)' + re.escape(a_norm) + r'(?!\w)', flags=re.IGNORECASE)
                m = pat.search(rt)
                if m:
                    s,e = m.start(), m.end()
                    extras["positions"].append((tkr, a_orig, s, e))
                    extras["contexts"].append(rt[max(0,s-30): e+30])
                    break
            except re.error:
                idx = tlow.find(a_norm)
                if idx >= 0:
                    extras["positions"].append((tkr, a_orig, idx, idx+len(a_norm)))
                    extras["contexts"].append(rt[max(0,idx-30): idx+len(a_norm)+30])
                    break
    return extras

# -----------------------------
# main apply (assumes df exists; demo if not)
# -----------------------------
try:
    df
except NameError:
    df = pd.DataFrame({
        "id":[1,2,3,4],
        "date":pd.to_datetime(["2019-11-16","2019-12-28","2018-01-04","2016-10-24"]),
        "text":[
            "Dow hits 28,000 - FIRST TIME EVER!",
            "Apple should give cellphone info to authorities",
            "I'm all for $AAPL and $TSLA today",
            "Lockheed Martin F-35 cost overruns are tremendous"
        ]
    })
    print("Demo df created — replace with your real tweets DataFrame named `df`.")

ticker_set = set(df_dict['ticker'].unique())
cashtag_only_set = CASHTAG_ONLY_TICKERS

# apply matcher
res = df['text'].apply(lambda t: match_companies_in_text(t, alias_patterns, df_dict, ticker_set, cashtag_only_set))

# unpack (keeps external API compatible)
df['matched_tickers'] = res.apply(lambda x: x[0])
df['match_method'] = res.apply(lambda x: x[1])
df['matched_aliases'] = res.apply(lambda x: x[2])
df['matched_industries'] = res.apply(lambda x: x[3])
df['is_quoted'] = res.apply(lambda x: x[4])

# -----------------------------
# export annotated tweets
# -----------------------------
df.to_csv(DF_OUT_ANNOTATED, index=False, encoding="utf-8-sig")
print("Saved annotated tweets to", DF_OUT_ANNOTATED)


print("Pipeline finished. (Counts/statistics skipped for now as requested.)")

Saved annotated tweets to tweets_matched.csv
Pipeline finished. (Counts/statistics skipped for now as requested.)


In [7]:

# 1️⃣ read results
df = pd.read_csv("tweets_matched.csv")

# 2️⃣ matched_tickers 
def parse_list(x):
    if pd.isna(x):
        return []
    try:
        return ast.literal_eval(x)
    except:
        return []

df["matched_tickers"] = df["matched_tickers"].apply(parse_list)

# 3️⃣ one row one ticker
df_exploded = df.explode("matched_tickers")

df_exploded = df_exploded[
    df_exploded["matched_tickers"].notna() &
    (df_exploded["matched_tickers"] != "")
]

# 4️⃣read dictionary，get industry 和 company_name
df_dict = pd.read_csv("company_dict_10_industries.csv")

ticker_to_ind = df_dict.set_index("ticker")["industry"].to_dict()
ticker_to_name = df_dict.set_index("ticker")["company_name"].to_dict()

df_exploded["industry"] = df_exploded["matched_tickers"].map(ticker_to_ind)
df_exploded["company_name"] = df_exploded["matched_tickers"].map(ticker_to_name)

# 5️⃣ statistics
counts = (
    df_exploded
    .groupby(["industry", "company_name"])
    .size()
    .reset_index(name="tweet_count")
    .sort_values(["industry", "tweet_count"], ascending=[True, False])
)

print(counts)

# save
#counts.to_csv("industry_company_tweet_counts.csv", index=False)

                         industry                company_name  tweet_count
0               Airlines & Travel     American Airlines Group            4
1               Airlines & Travel      Marriott International            1
2               Airlines & Travel          Southwest Airlines            1
4   Automobiles & Auto Components              General Motors           14
3   Automobiles & Auto Components                   Aptiv PLC            1
5                       Chemicals           DuPont de Nemours            2
6   Construction & Infrastructure            Caterpillar Inc.            1
7             Defense & Aerospace              Boeing Company            8
8             Defense & Aerospace             Lockheed Martin            4
9              Financials (Banks)               Goldman Sachs            8
12              News & Publishing  The New York Times Company           83
10              News & Publishing             Fox Corporation            9
11              News & Pu